**ST 554 HW9 - NC State Jupyter hub version**

Michelle Silveira

# Diabetes Classification with Spark MLlib

This notebook applies Spark MLlib to predict diabetes diagnosis (binary outcome)
using the Pima Indians Diabetes dataset. Three classes of supervised learning
models are compared using pipelines and cross-validation, and the best overall
model is selected based on test set performance.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MLlib_Diabetes") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 21:51:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/04/13 21:51:31 WARN Utils: Service 'SparkUI' coul

## Dataset

We use the **Pima Indians Diabetes Dataset**, collected by the National Institute
of Diabetes and Digestive and Kidney Diseases. It contains 768 female patients
of Pima Indian heritage and is read directly via URL.

**Features (all numeric):**
- `Pregnancies`: Number of times pregnant
- `Glucose`: Plasma glucose concentration (2-hr oral glucose tolerance test)
- `BloodPressure`: Diastolic blood pressure (mm Hg)
- `SkinThickness`: Triceps skinfold thickness (mm)
- `Insulin`: 2-hour serum insulin (µU/mL)
- `BMI`: Body mass index
- `DiabetesPedigreeFunction`: Genetic risk score based on family history
- `Age`: Age in years

**Target:** `Outcome` — 1 = diabetic, 0 = not diabetic (binary classification)

**Note on missing values:** Several columns contain `0` values that are
biologically impossible (e.g., Glucose = 0, BMI = 0). These are treated as
missing and replaced with `null` before modeling, then handled by an `Imputer`
stage inside each pipeline.

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
pdf = pd.read_csv(url)
df = spark.createDataFrame(pdf)

print("Schema:")
df.printSchema()

print("\nFirst 5 rows:")
df.show(5)

print("\nBasic statistics:")
df.describe().show()

print("Class distribution:")
df.groupBy("Outcome").count().orderBy("Outcome").show()

Schema:
root
 |-- Pregnancies: long (nullable = true)
 |-- Glucose: long (nullable = true)
 |-- BloodPressure: long (nullable = true)
 |-- SkinThickness: long (nullable = true)
 |-- Insulin: long (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: long (nullable = true)
 |-- Outcome: long (nullable = true)


First 5 rows:


+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          6|    148|           72|           35|      0|33.6|                   0.627| 50|      1|
|          1|     85|           66|           29|      0|26.6|                   0.351| 31|      0|
|          8|    183|           64|            0|      0|23.3|                   0.672| 32|      1|
|          1|     89|           66|           23|     94|28.1|                   0.167| 21|      0|
|          0|    137|           40|           35|    168|43.1|                   2.288| 33|      1|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows

Basic statistics:


26/04/13 21:51:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------------+------------------+------------------+
|summary|       Pregnancies|           Glucose|    BloodPressure|     SkinThickness|           Insulin|              BMI|DiabetesPedigreeFunction|               Age|           Outcome|
+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------------+------------------+------------------+
|  count|               768|               768|              768|               768|               768|              768|                     768|               768|               768|
|   mean|3.8450520833333335|      120.89453125|      69.10546875|20.536458333333332| 79.79947916666667|31.99257812500002|      0.4718763020833334|33.240885416666664|0.3489583333333333|
| stddev| 3.369578062698868|31.972618195136228|19.35580717064478|15.9522175

+-------+-----+
|Outcome|count|
+-------+-----+
|      0|  500|
|      1|  268|
+-------+-----+



## Data Preprocessing, Splitting, and Metric

### Missing Value Treatment
Columns `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`
contain zeros that represent missing data. We replace these with `null` so
that the `Imputer` stage inside each pipeline can fill them using the column
median — estimated only from the training set to avoid data leakage.

### Train/Test Split
We split the data **80% training / 20% test** using `randomSplit()` with a
fixed seed for reproducibility.

### Evaluation Metric: AUC-ROC
We use **Area Under the ROC Curve (AUC-ROC)** as our primary metric.

- It is **threshold-independent**: it measures discrimination ability across
  all possible classification cutoffs.
- It handles **class imbalance** better than accuracy (~65% negative, ~35%
  positive here).
- AUC = 1.0 is a perfect classifier; AUC = 0.5 is equivalent to random guessing.

In [3]:
from pyspark.sql.functions import col, when

# Replace biologically impossible zeros with null
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for c in zero_invalid_cols:
    df = df.withColumn(c, when(col(c) == 0, None).otherwise(col(c)))

# Rename target column to 'label' (required by Spark MLlib)
df = df.withColumnRenamed("Outcome", "label")

# 80/20 train-test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training set: {train_df.count()} rows")
print(f"Test set:     {test_df.count()} rows")

Training set: 617 rows
Test set:     151 rows


## Model 1: Logistic Regression

**Conceptual description:**
Logistic Regression is a linear classification model that estimates the
probability of a binary outcome. It models the log-odds of the positive class
as a weighted linear combination of input features, then passes the result
through the logistic (sigmoid) function to produce a probability between 0 and 1.

The model learns a **linear decision boundary** — a hyperplane in feature space
— that best separates the two classes. It is interpretable: each coefficient
reflects the direction and strength of a feature's contribution to the
prediction. Regularization (L1 or L2) shrinks the coefficients to prevent
overfitting.

**Pipeline — 4 transformations before the estimator:**
1. `Imputer` — fills null values using column medians (fit on training data only)
2. `VectorAssembler` — combines individual feature columns into one feature vector
3. `StandardScaler` — centers and scales features (critical for linear models
   sensitive to feature magnitude)
4. `PolynomialExpansion (degree=2)` — adds squared terms and pairwise interaction
   features, allowing the linear model to capture non-linear relationships
5. `LogisticRegression` ← estimator

**Hyperparameters tuned via cross-validation:**
- `regParam`: regularization strength (0.01, 0.1, 1.0)
- `elasticNetParam`: blend of L1/L2 regularization (0.0 = Ridge, 0.5 = mix)

In [4]:
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PolynomialExpansion
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

imputed_lr = [f"{c}_imp_lr" for c in feature_cols]

# Stage 1: Imputer
imputer_lr = Imputer(inputCols=feature_cols, outputCols=imputed_lr, strategy="median")

# Stage 2: VectorAssembler
assembler_lr = VectorAssembler(inputCols=imputed_lr, outputCol="features_raw_lr")

# Stage 3: StandardScaler
scaler_lr = StandardScaler(inputCol="features_raw_lr", outputCol="features_scaled_lr",
                           withMean=True, withStd=True)

# Stage 4: PolynomialExpansion — adds x^2 and x_i*x_j interaction terms
poly_lr = PolynomialExpansion(degree=2, inputCol="features_scaled_lr",
                              outputCol="features_poly_lr")

# Stage 5 (estimator): Logistic Regression
lr = LogisticRegression(featuresCol="features_poly_lr", labelCol="label", maxIter=100)

lr_pipeline = Pipeline(stages=[imputer_lr, assembler_lr, scaler_lr, poly_lr, lr])

# Shared evaluator (reused for all models)
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

lr_param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

lr_cv_model = lr_cv.fit(train_df)
print(f"Best Logistic Regression CV AUC: {max(lr_cv_model.avgMetrics):.4f}")

Best Logistic Regression CV AUC: 0.8235


## Model 2: Random Forest Classifier

**Conceptual description:**
Random Forest is a **bagging** (Bootstrap AGGregating) ensemble method. It
builds many independent decision trees, each trained on a different random
bootstrap sample of the data and using a randomly selected subset of features
at each split. The final prediction is determined by **majority vote** across
all trees.

By averaging many diverse, uncorrelated trees, Random Forest reduces the
high variance of a single decision tree without significantly increasing bias.
It is robust to outliers and does not require feature scaling, since decision
trees split on thresholds rather than distances. Feature importances are a
natural byproduct of the model.

**Hyperparameters tuned via cross-validation:**
- `numTrees`: number of trees in the forest (50, 100, 150)
- `maxDepth`: maximum depth of each individual tree (3, 5, 7)

In [8]:
from pyspark.ml.classification import RandomForestClassifier

imputed_rf = [f"{c}_imp_rf" for c in feature_cols]

# Stage 1: Imputer
imputer_rf = Imputer(inputCols=feature_cols, outputCols=imputed_rf, strategy="median")

# Stage 2: VectorAssembler
assembler_rf = VectorAssembler(inputCols=imputed_rf, outputCol="features_rf")

# Stage 3 (estimator): Random Forest — no scaling needed for tree-based models
rf = RandomForestClassifier(featuresCol="features_rf", labelCol="label", seed=42)

rf_pipeline = Pipeline(stages=[imputer_rf, assembler_rf, rf])

rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 150]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

rf_cv_model = rf_cv.fit(train_df)
print(f"Best Random Forest CV AUC: {max(rf_cv_model.avgMetrics):.4f}")

26/04/13 22:11:29 WARN DAGScheduler: Broadcasting large task binary with size 1001.5 KiB
26/04/13 22:11:30 WARN DAGScheduler: Broadcasting large task binary with size 1376.4 KiB
26/04/13 22:11:33 WARN DAGScheduler: Broadcasting large task binary with size 1301.7 KiB
26/04/13 22:12:09 WARN DAGScheduler: Broadcasting large task binary with size 1357.0 KiB
26/04/13 22:12:11 WARN DAGScheduler: Broadcasting large task binary with size 1298.5 KiB
26/04/13 22:12:47 WARN DAGScheduler: Broadcasting large task binary with size 1337.3 KiB
26/04/13 22:12:48 WARN DAGScheduler: Broadcasting large task binary with size 1280.1 KiB
26/04/13 22:13:24 WARN DAGScheduler: Broadcasting large task binary with size 1011.1 KiB
26/04/13 22:13:25 WARN DAGScheduler: Broadcasting large task binary with size 1390.4 KiB
26/04/13 22:13:27 WARN DAGScheduler: Broadcasting large task binary with size 1323.4 KiB
26/04/13 22:14:04 WARN DAGScheduler: Broadcasting large task binary with size 1012.6 KiB
26/04/13 22:14:05 WAR

Best Random Forest CV AUC: 0.8277


## Model 3: Gradient Boosted Trees (GBT)

**Conceptual description:**
Gradient Boosted Trees is a **boosting** ensemble method. Unlike Random Forest
where trees are built independently and combined by voting, GBT builds trees
**sequentially**: each new tree is trained specifically to correct the prediction
errors (residuals) of the ensemble so far.

The model iteratively minimizes a loss function by moving in the direction of
the negative gradient — hence "gradient" boosting. Each tree makes a small
contribution, and the final prediction is a weighted sum of all trees' outputs.
This sequential focus on difficult examples gives GBT high predictive accuracy,
but it is more sensitive to overfitting than Random Forest, making tuning of
iterations and depth important.

**Hyperparameters tuned via cross-validation:**
- `maxIter`: number of boosting rounds / trees (20, 50)
- `maxDepth`: maximum depth of each tree (3, 5)

In [5]:
from pyspark.ml.classification import GBTClassifier

imputed_gbt = [f"{c}_imp_gbt" for c in feature_cols]

# Stage 1: Imputer
imputer_gbt = Imputer(inputCols=feature_cols, outputCols=imputed_gbt, strategy="median")

# Stage 2: VectorAssembler
assembler_gbt = VectorAssembler(inputCols=imputed_gbt, outputCol="features_gbt")

# Stage 3 (estimator): Gradient Boosted Trees
gbt = GBTClassifier(featuresCol="features_gbt", labelCol="label", seed=42)

gbt_pipeline = Pipeline(stages=[imputer_gbt, assembler_gbt, gbt])

gbt_param_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [20, 50]) \
    .addGrid(gbt.maxDepth, [3, 5]) \
    .build()

gbt_cv = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

gbt_cv_model = gbt_cv.fit(train_df)
print(f"Best GBT CV AUC: {max(gbt_cv_model.avgMetrics):.4f}")

Best GBT CV AUC: 0.8181


## Model Testing

We now evaluate the best model from each class — selected via 5-fold
cross-validation on the training set — on the held-out test set.
Test set performance gives an unbiased estimate of how each model
generalizes to unseen data.

In [9]:
lr_test_auc  = evaluator.evaluate(lr_cv_model.transform(test_df))
rf_test_auc  = evaluator.evaluate(rf_cv_model.transform(test_df))
gbt_test_auc = evaluator.evaluate(gbt_cv_model.transform(test_df))

print("=" * 48)
print(f"{'Model':<26} {'Test AUC-ROC':>12}")
print("=" * 48)
print(f"{'Logistic Regression':<26} {lr_test_auc:>12.4f}")
print(f"{'Random Forest':<26} {rf_test_auc:>12.4f}")
print(f"{'Gradient Boosted Trees':<26} {gbt_test_auc:>12.4f}")
print("=" * 48)

results = {
    "Logistic Regression":    lr_test_auc,
    "Random Forest":          rf_test_auc,
    "Gradient Boosted Trees": gbt_test_auc
}
best_name = max(results, key=results.get)
print(f"\nBest Overall Model: {best_name}  (AUC = {results[best_name]:.4f})")

Model                      Test AUC-ROC
Logistic Regression              0.8765
Random Forest                    0.8655
Gradient Boosted Trees           0.8173

Best Overall Model: Logistic Regression  (AUC = 0.8765)


## Conclusion

After fitting three classes of models using Spark MLlib pipelines and selecting
hyperparameters via 5-fold cross-validation on the training set, we evaluated
each best model on the held-out test set using AUC-ROC.

**Results summary:**

| Model                  | Test AUC-ROC |
|------------------------|-------------|
| Logistic Regression    |    0.8765   |
| Random Forest          |    0.8655   |
| Gradient Boosted Trees |    0.8173   |

**Best overall model: Logistic Regression (AUC = 0.8765)**

All three models performed well above random chance (AUC = 0.50), indicating
that the selected features carry strong predictive signal for diabetes diagnosis.
Logistic Regression achieved the highest AUC, which suggests that the
relationship between the medical features and diabetes outcome is largely linear
— particularly after enriching the feature space with polynomial and interaction
terms via PolynomialExpansion. This allowed the linear model to capture
non-linear patterns (e.g., the combined effect of BMI and Age) without the
overhead of a full tree-based approach.

Random Forest performed competitively (0.8655), confirming that the data has
structure that tree-based methods can exploit. Gradient Boosted Trees
underperformed relative to the other two, likely because the sequential
boosting process is more sensitive to the small dataset size (~615 training
rows), making it harder to tune without overfitting.